# SemPointer: Kaggle two-T4 backup evaluation

Attach the existing private Dataset containing `sempointer_qwen35_frozen.bin`.
Select **GPU T4 x2**, **Internet on**, and file persistence if available.
This notebook includes its launcher; no second code Dataset is needed.

Run all cells with `RUN_MODE = "pilot"` first. This runs the original development
phase and representative pilot, then exports results and stops. Inspect the
projection before changing the mode to `"run"` for the full suite. Pilot responses
are reused. Sources, selected questions, controls, output caps and metrics remain
frozen. Precision is NF4/FP16, separately labeled from the L40S BF16 experiment.

`HOURS` is an allowance **from launcher start**, not from Kaggle session creation.
Keep it below the session time and GPU quota remaining. A new full-run saved
session can use 9 hours; the default pilot allowance is 2 hours. Reserve time for
setup, exporting and saving results before Kaggle's 12-hour limit.

Download `sempointer_kaggle_results.tar.gz` after each invocation. For a new
session, attach the previous notebook's saved output as another Input; setup
automatically restores a single matching archive. If several exist, set
`RESTORE_ARCHIVE` to the chosen archive's absolute input path. Persistence alone
is not a durable backup. A session stop means partial coverage, not completion.

For unattended work use Kaggle **Save Version → Save & Run All** with these clean
cells. An interactive tab closing does not provide a reliable background run.


In [ ]:
RUN_MODE = "pilot"    # "pilot" first; "run" continues the complete evaluation
HOURS = 2.0           # e.g. 9.0 for a fresh full-run session; use less time if needed
RESTORE_ARCHIVE = ""  # optional absolute /kaggle/input/.../sempointer_kaggle_results.tar.gz


In [ ]:
import base64
import hashlib
import json
import os
from pathlib import Path
import runpy
import subprocess
import sys
import tarfile
import time
import zlib
import torch

WORK = Path("/kaggle/working/sempointer")
WORK.mkdir(parents=True, exist_ok=True)
ROOT = WORK / "frozen_bundle"
PYTHON = WORK / "venv/bin/python"
RUNNER = WORK / "sempointer_kaggle.py"

assert RUN_MODE in {"pilot", "run"}
assert 0.05 <= HOURS <= 10.5
assert torch.cuda.device_count() == 2, "Select GPU T4 x2 before running."

payload = zlib.decompress(base64.b64decode('eNrFPGlz2ziW3/0rMOnaopTItJ1rq5TRTDmO0vG2Y3tspXtnvS4WLUIW2xTJ5uGjvfrv+w4ABC/Z6emtTVe1RRJ4AB7e/R7ww192yjzbuQrjHRnfivShWCbxm60XL17M7pLt2VvhB35ayEwskkwUSynC1aos/KtIinO5Ok3CGD9elXEQSVd8kgu/jIqxSMMoKUQSRw/u1tZxIsJ4ITMZz6XIyjiHDwAnTbLCFd9ymRNghuHkQt6HeRHG10Le+lHpFzCwHwdiJYssnOcuzm1rkSUr4XmLsigz6XkKGrSLk8IvwiTOt7b0u+w69bNccp95EkVyTi10p4OkxEWMRMCzD8J5oTsv5nER6Yelny+j8IoB8Tt4dGFigKTC1/BO/fmNfy2Pk+IzQA6mWZYA8FuZ5TCqhvVrXv1OcgaZ+gXCr+AUS90kX5ZFaCaSh9exXz39FoWFfGMey6s0S+Yyz82bB/Oz8LNFGEnzKFdp7Tlcya2tX07OfhITGn/g7MBariO5c5dkN7ApOzl04V13hlvn/zyfTb+eQ+MLx/owEo766SWLBT5erV6/816bX2/Nr5VfzJcy8K7K4FoW+HpRRpE3T6D3feFcbm1tzSM/z8WUiQFQ+NkPI9j1wUc/l9P7uUzx5XC8JeAf0MY0n/upJJICrP4uYyS5GGYFZJcFYexnDyKV2ba891cpkLHE/RGZnCewRQ8uUtfWFpCCQMR4uOcD3Bg1wF1YLBkz9NJNUhkPnOzKGQo/F3mRSX/FLfFfJoE+Y004LkEMwmuZFwNuCuvNl/7rd++dobuU9+rbUM1APeLKpRq/AZE7D5Ca3KBcpTk3HokcttO7kQ/5ZJbhs4xz5BQ/n4fh5LMf5XLoAj8mgRwMu4aGyQW0RMMWk+MkbsyCho0SP8gHFkqwq4e7B5BFuLCxRaydD4ZCwgQ0YDUiMPoqnKsx7SXjG02OBMa8dYGxZVy4q5sgzAb8YBaMQ3nJDT1yF6R2AEQ9cR+9vFwswnsC6vJv8Uo4brFKnaqHe5cBe/GC2ngO4wAGnbzuxDBC++/YhpXJNPLnUq1D0Vkm5e+yY+HduKuoK7S2aSj+MuHO1XfaKT8ETP+MH0gQDRbOQRIvwusyI2YS86UfX8sA0C8eEdD6AwDNQegyCxF/ojAGJkLpADwkSoDoi1jeiaQs0rIAOgX2gS4Paqm4u9U0OjZWrRwYLlw8eCz5B1mSFJq+4KfecnpNb1d+HC5gbvCF1k2tdoSj37u4PWoKgBtQBVUXmLai8Meb8S1ps5vRLWyfaeLCLq+QNqHrDWLTgSV7YeCshxfj17uX+Eq3vdDfLi1eb2La+czix8wBBN38Ji9XYhXmhFenIuUcZahZEfKlAz9if0WknAJ6ZTCkadff2Su4oH4eSpncudTrubTGeGUNkl04+NK5HOFPJYUueYysDjZL7gAgUHP1KpC3MkrSFRC/c8lDYEfeZHtyNHCNZnFfiOWAzpLoFgSQG+ZeJiMgyFtgtIRmWH0d4uY1hDFuhh7lSYJX+5AnZTaXO1rut3ZjrBhA7YqScXrJimTzFJSthwgZ4P9GAvWizBTdXmdJmeJWWqbEIAK+HRoMQS9EC3auJs79LuDlhbPwV2H0AEh1/RSUC5L5XW1GF2yMcTuEhbJeBgMGYm1g7f0Ft4fdBq0wifzVVeCL+/H9hYNkPLzgdYzHry8v1VJTRIon4+swlgNGnlomywxc56NZAuj/aOHGi7egu1cgFEDRFQ+ghyei9W4AxBMCMCQG1WgiXkCzF6wXUNE442dABpEzX7oL0EDF3vuN0EYVtJs7sAhpj3Bz5iQMJ/xnJKJk7kfMQR5ar0qZENwJD3elxhvaU/xXgRqYPNG1IZcrCf+Hxv4C7exQy+tcc3eNs3iT3DkatAPuSZyy9wSLON9iw7LKZkpBoodosKI1Gknkc9IXqB0WpABiATQEeiQpcyYVxTf4jycC6FAz0mqvtpoaUXNDTXnQHpS5p3TOgBXESOugkWFKrS74/XPtAC1F+2VnhS8wmMEuRY2j9B8L5cb37zREWCA/oOUEkNXymnJfGVJXD4WEfa4JUJw7ELkjjU2889udjN+8c9MHp77X1TA1dlav3UBqE1Abg01JrRZoLDd0A/mVPT2kMgWzPr5qyzYUN1btatv/iGsaV0K+ju3hM7ZsrWiH5ZgXJUlKegSIBVpFaMb6QQRrB9u4SFIvhXHILkZ8ePMot+1bcALOyliUsbaONFdgp/wD6w5ycXKCBmxa3ElocC3B02AHlDwJIjf07aqt0u7dPErKwOPpjgTvX1d7l90X3Q3mdY7W2QY7KQdWlZ4ym7VJQS/ZQqJGSRbCykEuMQagpSKhf9CfKb2tt0QfKUNS0nNw+c2WtrgqZIowJ8FriSflYaK4My9pqUXmxzls8Ao8ZN3sKLkOi/yU/dgk2zI92Bn8HMZAT9xo0Gg7rBMg0oTngQCOPG+AmgRtdtAiYL2B9s5h/jJvdFGrGbBoDvPYjweqoevHD2ivgPBVvKDapEkexovOZi3Y+h/SdFIYEAtaVA1EEK4m23vwO4oMuI7JVkL9M+oRMNcpMKOEO+zDNsO2yBOUEmLtA5MvxlfCOEcVUEl+pz1zLaxpis1Nmb1lohk0SKtzQ3A+ZkNevmT92bG0vARfHdZuepi2XVvWYyn8pWEqbMKgrRZnb43YX5VgwKPeA0UZhfOwEJ9PweZQ43SgiqZCosetsO6xSeCy3+QxHoGhyF1sgVD9SI+2oMmtVnvELEkmGXi61eAlIgtRnFCcJO8hH/X1wmG68FLNTmD3g9NQYzjlUvQQR//IW89AEi62uYatFv0AqyVEOx2rackUHiB3Sb5577SIobfAKp5+7+egYtAF6SP7x5cvNTnSBIYj4cSwqb+VPhDJ7zBjIjdn7NSozTI+m/8c5ktPIf269LPAGbPd4FyRws4BsDPe2wTDB/sCpb3n3/phhLFZZ8wrQceq4/O6wqglsieGg/kzIppmBAtjQT+guMAInRcL8Q39YDeqhSqSO9jkYkCaqMydIZswyGlOl33aDvgZCNwJUFS14TAeWKf+nfALjLUUuG0+rCZwhor0OjQcrLpCATVqKDdoUEcCe0CW8na96fGPh8dTaIkqr1LBOk5R6eOReHTIhnDGuyOhMTHGaEIMpIhLAmkDv+Dl4xrDqH5WhIBalEH0bj3cqrE6GSG2G0DgO0deDxl7/pz8bJ4HzyJDnWEJsTRDqAvn4Mv04KfTk8PjmXgkpxxtsDX4ydh3DZ0XUZkvlWXLUa7soZrL3RJsM8bFhVr3pfirZX/V9x2IhM34BNRiEodzULF/mxjDTWyL97uoNDk+wFY2mjbns5NTpytAZsQM2T5lGqBYYqRPALUgzoFlFXRHG7xPQGenUvfGpXQI/yvA/s1Wawoo22zCwSAS4aHPUq22uJ//2//YuR8/sZAaczb2CLDet0mqpSJdVA4OKsFIFozCNqT3CmXw5Vp6pnEzPmjTKv0eNsPo9JYDjBT2F7UMAMpveF/NtpcHG9A3UYeWNPR3snAeKX4Bw6BFgszgeeuxeIQXa4sMnrEiFHFs7aPojCymacufhoA1LbvFWcP6soy0fpGl3e7kV7AAYZgVyE2QCAPtauVqAfBX719g+0tKaG0MfPF2hGobreBXFcxS4eHctGKBpUUiy7BW4MOagA5yAZEXGDpjcCr+nhQ+abmLKlwZVqEEHFPG5YptF7Nwa5wgIfwODAZQFl2gmH5cXw4vwsoq4uwnWm3FoPJW6S26Hc6lRSplELabzu+kR1+azTFsiEvINoRpUddyVI/kGcZgaT5o5Dc+4JIUjgECIBinMbTsOxrv1fMHhM+8oo2D4fLmfvrmdceItE2wxF23RiCdYVM7TKEIoB5CbauCtpH/NcxzTHBU28roYog0uCNeiRrgusTnKb/C9MD9oHsiQ/FS7Lnv6qvMdYSXnurxEEcxo9Qs6+XA/DFQw5j7onFQNUH31jSgaahGOnm3i8moN7u7dT3i5P5CFg9gI2L6BsxM9x0ZnnmIVskvSYZZiwZefgOOInmbShOE9udZAh7gVVIsVQAG51cf6t3uv4kliGLQSquRSMFwANl3C55VGIMHlfMuSzTNAX+uOIznURnAe1iayMN74WernMbUE3AbhrUz90t8DzM/ZczgLDHg+kFQMjorU1wGThBzXMDrOeVzwRvjsCruN3gVHAjNKU2Qu46OLIHnh0lVH0x7mPeAfbgqYHQe+2m+TAhhq7BAEOf/OEKvG4sDMOqAC8Ihftk/AvMY1ILEkgo/hh7pA36IEB+fPlahI5VfUxEeNSJ9UbNAeU9vLT1fJeI9zt17uNioyF0w9Nzr33lffhAnIM7QUE4yTIl/+ogpQF9wgBWA5mHAKcDcv5XBdjPV54oz/05BUgZ3TlyfZhKlP1VY+BEq3wdeGLAvI+SD8G+TMBBBia40agVGbe5WKXZdmeDO9Aw/6ZEHMMIivJ9YC91mANsqCz/PKJ9TcZtZ5UQjrpYAtlO+1Qy4VIIz/AYCkPXdGHDIihAQm0eYCZu8oZGLRV3i6JSYlY1R25VdR8nVwHnpdMVzdC5MJ6vt3JhNBM2AlcmsodsHc+cQlX6TP6zAqrzhlw4FwRySqQo6klCRb4pWESS0elyQWjliaTBwtkH4oBuxnS9X+NeNkvkN/SCkDmsdyecjQ3TTOIOqfYiyEDAcA3Vg8BKGciwKM+80CaoX643mMpLpwCogQK7RPMLVNLad3BMxwRIVEF3taAFQGC8Vy0G4MMKpMyKQ6Y5Be+dW2iUJiDI9re6pmCQFDabIf0iyQAkl3b+zO5G7qiJyYVkx8Bk4f0hBY06J/h09k0mWwLaWWcj+HnEaZW3GvcjuhMyzpf4g34vxxq1SCaQrf35TpgNs3x1PLRauHwQK9kjvwUT9HW5Am1vGzBatNvUyho7R2LTvHcsIDZP/UsKnpurVO51fLlcrPwt/B/sviuo6piePwOmBzsyBbfKb5qRin9AuxvwlWxP9J9V2Ryyc67R8DNc7JEB2GrUXbFWTVMHEyeC1suzY5t8AzUpP2EGSdS9E4+ZwrcpE4cGt5UcqX4j0g+0dUTeX3ys1ptQ1uz9clIjhJJXMTBaiyEpgS3HOewTrCRIyPmOJBTSCDTAF7D/OT46PSLNjT4m1ZcaUQk8LA1R1del2rMs1E1cZewx6jqs4Uy3c0sJUy1LuQLwRSRviJ0wz7c5KZbZc2L41NLaDI2nJ6gqDdMaS4Twov2R5pdxp03JHaa8G8Y1EnbYc3jivQaJVjqzMW9ELZLq8Eb0g57VyXKsIEIXmeqfYoGgd6+O/65pffE167kI7YZZ/ZHkwpkqodw93HgmWedZCv4+FLPcJk01pFeMiu8VU7zQooqX4VAGbvXhNVfCb10ebSrkPxkiTxh7VboHxfga7uI1muyhjsuAXxIrFXULsqMQZGtVXmI1FhAHvO+PG7tvDDdcqPMmZqv3r60xew/aQpLA8/HZqyti7aPAhwAYyKINhGmlZal4MOxpfoWXQ0IngtMkVtB/bGrbZFxpa8Ytncr1VQdnP/LhNtYWw8dU2i3ptlX7hYS0cWsk5OHtgXOzPZvsHX8Sn/dn+x/3zqfi72D8XdyhTRqLP9hgO+4AjZrQzvXDOp0fTgxl43J/PTr4SUDfyibc6t8PMyjk4m+7PpmI2/Xoqfj6c/iK4G04NQwCO+HZ8eHIs9o+OwCL7FVyPgRp7aOW2mIgk+V9eV5JKRzPhS6vXvZ0V7UiJ6imDZs+xWKPWn6Vq15ApUP2WLpWxtSRQRZ0Z2lqFqolyiqMSeM3mPbK/PUBdG6h4YRFRNLwtMnbOpqcnZzN3pYQ/NbVrcPmFVWRsrCvnB/E1CfwIDK94+0rG8yWo6RvLIEJR8YP4iWxxlCd4vsH6OqzZZbxuUwtHNf2GwHmyWAC3bpUUXXD6j5I5VhoSn/35HAgGI4xUAB8WOZgAVBWDz3G5Sh/wR6i0hpM++FmW3BGo5EbGYBcioDYbOMvyGsMXC8CDtyyvqJA+8rcxwax+58ttTCr42Tb6SrHGx9wvcz+ChvHtXq2etUYE1ZovcJkY9lcHGgZ1m1dF5rvOQGyEZiwahX3+rLBvQlCejv3kddPYbIcd2+2WiSqK+7iumU3PVcLPkaZ9ariexYZpXFC7SxX9HW59n1z9415bcNUtpetzesQI5u6lHX21ZGWv0/ZCSV9VZuuFAYthJUp/+TI9m+KJFHCOxOGxGPx9ZP4bihf9YPePP2mjDbs5VDsRgOef34Qg+APwPX48O/l2Kj7+0x77y/7Ph8c/ioOTb8ezwafD89nhMUyOxx9O3r8YCXWGZbjeqpOm1ieIlWEvaapzIx4H91XwVxFnlVww+ZMqXLgvln4WiJswisScYoAYNUTgKZA3+BZcK4mmT1jFPCk8l1l5QRMqVBMhQ7qe4TB5BZ2Fqqc6fg/Tdp7HDnpUyZ3HxRhzOjqtwpmb70narP9IioRizCGXG6uUwjirYuv9iQl7RzFx1p+fehZbI/328vRmfv1zeXXBITKqohwNG4yprR9d+Gkxn9NXOgbcPbEP89QqNDssPsptVEzGeR/au/5gjtqCi1bfdp27oV08WjPiHBTFKQlCdwIQZoXEwYxtEwX3pgyVyVlTULElQNZrLBRDgtQiob2azqDfIkqSzFsheeXlapAxA1hVXytkg91hfVLDbhgqjaSAhFjr/S/C4GCDl5doQnFp9bOnpbjYBawozhlodryg7bkcgbQZGttbz2BH7O3u7g5rDEjSSbfEWioUFnbFi/qxbkhb1bNRG67STKY2vCly0V8LuJrYiN0NZoGqpG2HOEwLcziHI+fD7yn2VvSeW/W4HMHQH5ymQ64/9AgU81kF9TEzUJWRG6gjFeP0wtwzvl1jbiqlbB1sacrRkQitYpHAFsR12llb7kgW3pIugo2utkSHZHiqnjpENW4fq8I0C22wMw47S1/s2u+xtdPQURfhxQs6ZmqOHntPF9TpOH1VAzt2jj+/FXcyvF6iduys/qQ4XpzE26YQkL6A3QCE0V39x5gd8xEiS+2NL+4Jpfd1fWbpRSQQ+gpPlz2oqVUP9IBslBjUwLb2EWQnUhRWS7IXSIU/YA48Ojpf6oxJ9ll6eTIRi4ZsWbPJUJ+JFihOe1w3TdKBJophe1aaXOjgDZ3uU5/4yF4zQlXnvWb0UHetqmGuMb9X9zRgSzO2RZ52NWoFSM8Oedt1gaY0bz1s++P/3y7M/5GLMiIbF6Y2ebfRW2E8VP4KmXQdhhAjc8QewMthzRsxHoOuSbVENm+00Wwvfjz9Bhji+srJY73MbS0eLdNXA4O3PMn1i5pKc8T/6PiRGsWcugZ6ni/DKACBNeDyaVNOYZ9a0RWC+oQ0F2vbZMnHy60XJtmDYFUB+oYUghq2WfuHQLEGqk05qrAsSq4dfSreh129KhcgfoGEJ3vDDlgGv/xYbyLj2zBLYjx7xookAXXI74C4D7592vd+Pjw//Hg09T5Nfz48mJ5jeD0bhFiYffL11Dv+9tWbfTmb7n+CL85rZ2MRpTM7+Wl6fPhf07Nz73T/bP/oaHp0eP4VOi6wOh/56ctn78u3j97J589Hh8dT+LKHb2dn+8fnn0/OvmJP+9u6ETZfrVBRwI6AlarIFKuxMXdeYiWUzhZ7pNc8b2idgcVG20olclNYZd9yoClvkGqqJBjBUAca1BciBvWKv1MNqv2VXmwcy5TQci9DnAQPyXZb1xrj54qOG+cIDBUbmjA3WbinRFEKhQgnQAGh71CAR5llE6v9+ezTybcZHu+6nVhkZPF35NNBcqvMjCaWhzEdXDTxqKqOmXMkaYJncfT5JitTomffkIMxOVfNuuZ2CdvdNs3obxPxZrcjgk8F2U2VVK+/bpcB0woBdHO02kpUdd4Al2PcgY1LImu4SrSouuIiKefL9sraDTfVaVs7YD3QQam7DpzZ5eGv9rg43OoHm4Sro41iG+1u2/r8N0D1sNu9xqXXUDDu84WxRYMo+p1galzIDPwcDL08p3CcKCePpEzBttj6jjm24rdmAnd+iGF0VrJ7uw2hywFci5dm3HJ6n4IHEfTAxDhWx3qq8Wrl5kyArA7ReEfHe/dpwuPizTMWV6p8c18nBHFjUyzvQ6PDVwdCBBUV+njFDCURKeamKcYVzlNV9c6BKear1bVl0pwu+WBOzoEmfGnpQR3Bo1JDrGnsS5c/YzOfS2hPElgnVTyPMv4YdTxFIX1UwiYt2RsUWCRrocscqXJhKlcAJhVe8sTqS1EQGdJ9crjXNFK1es909Inw6gloNRMuiavMIrJr8Z1FA3jRFJ4Ym98M8H8j9ebo5OAnb/qfYDFaz8cfKzxZl6LUr1OhxfDvGu9VjhcHy9D5A/ZTMa/vuClABfEB6Zgx5xu1Yn1SWo9ieW2dJ4DxzA8dlpuXge/yxRAeX2BAh8pfC7r+62FgtcKAhm7pp/4VeI/FA9hD2H7w76N3HTVDw6fWNVsCT+m7zjL5WxniyUx578+L6IFqD2YlmrFi9laAJ5Dbpz2MlaRJiB9gDnSTFt5QQycQMc1eYZ/egTt9y4GGWklGdRDegrdjJQF1uVXtThrzle1HfaXLU0s/DWNMwVY5hR0DCAXNStXHJ5m+KwjvgACbEUOlpV02i3ch3Wt/tz5xPuNJ1eYyzpMsd6mxfWMPsRDHWdDab1wIUAPXvhNAFddi8JYAXzgMylv5qXPpcoYCazkrm5zjRVhi27htwmKckbANZOveCcsbx6IQnPD3xLYiH68WwKJ+3iNrocb477K4m0Qz7icnjKAxCvQYJojlqPV4dgxNhc+6BrWjYXi4epHiodbmAVVh5MlYyRI6v8h5c2dcZdA7xwDdiaGqbi7HXUb+btcxdcJC5z6eP+DxagoEnoNc9CM6OwCs+4GPKcAv7eTCtj6g642bOS8zFPN4ZoDP5m7z2Vw+BAr+BHpsgeXaqdBSd92RKRrqb95ZpNZFac3Ad41M29FvFfiuvBu6QdDFgsTXhshGVs1d69gAXiZS9W/b8apuVsmHdtAbvVyuPY9zmRWD3Q6X047pqVOeE6WsXzHvLQE/+cs373d3xbbYe71r4+M2TMoc2AZxkDVCYpycpbwpndIc4KWJJTiLiwylR10oPu3NkDlCECjExjcwuueHPx4eg5tZPc6mZ1+bwbbmRC8YECX+uSP/MVO05z3sL8jg0Ctmhe3aSeCk3SpnuYRfyyQKPNPYugOkkXOomvS7aOyIOp+q67/G4s17rfXzhxis7CKcm4Q1iBucg8QcdxAGfIYQ1akSFq6z2YntiIhd7IJwe2MFwf68tQg+9oknYejkCxi4dLI0ipI7H1zGD+rckDoXg9fhNY/FPLUe8nnqabumSIAd3Ht6B1W5q3k7bBa3YTGyp2PQDejN22Ba2Ks6b0CgPudqtbYi2W/+pRPujU06peyyEd1ABrqsrDqxVT+fxqeRqit8FAbAYtxIdJ0EN9oDknvbR3J/erV8LYHwp1bNN8t+W+XOb1vlzr18Q1uCFqLawg8AUpdq0flA6xBeJvBaV/v0Agl5GfwBfllJH2/aDMjS/L7qm2a88VcjPzeefdYjjjZVovUBB8NPw/TC+DaZc/Jdn9ZELYDJ+t2KurabXmodtOK6iqN59dWImiCqN10sZd1lWjWsLjTdEFzEAI42x+l0Ek3A6SeUw6OTmTg4+Xp6NJ1N8WinvmlID/tBMTFZYQS2SEythTmjCQaQpNuNFKfb58C+n4zU5D5/OzoS05/3j77tz7C493y2fzY7PP7RFfuVTUKXJOFpP5ndqiOpZYz2YY6BpUri6EuywUkutPb4Y8LmfZewaUWO6pbJyJgaHElqmB3dhTMkZzqtEA1g+HRY0eQkO48v9dHFga7+5coeDCmQupD5mCxnhNmPu05k1I/Gdp/Y7ZnOwjmbnn87mon9s4Mvhz9Px+JRdVw/QVzajkjuYiyb2sEDs0AGGFBQU1GRQBnT+V+kEFWE3EMjHMpCqYFUcWtftUB3mWccaaDf7n52XaLxckpf8IzcPAvpSoyJ5wXJHHxJqyceXAOMcJeBSvggupcJuFr5RBUb6FMVl9URcvVhIzCy1rHsBK+2pPuPqv6v3Y1d1f3vqu9p7eppHUdhM1O52JtnYhJgG8A1T2VuglflzZ4BULV9Eu+RDW5ja5P6o+Z4IUy1YywxljJKJ4Yozr+dngI9n28EaiXu7A37A5BqWb5qipsgqQtTFUD6gyDJsb2tXeLs7r4Tf51Y7iA+7e267+zDUQRFckRL0SFf2nYlze2MuwgJrULsPKJfV2D634kHaC6Mmm4wp7D2Uqs+faaySjCN/6Dfa99XacWfjZVnDYdyqIqDqd/VZUH0aGVXO67ibgbHQdCEeO6Eb7IhXe55iAXPU+qcZNBw638BEpdsHw=='))
assert hashlib.sha256(payload).hexdigest() == 'd57df846362f26b2bc1e8747bbd6056132b2b3b54892848eaa91564349839eb1'
RUNNER.write_bytes(payload)

if not (ROOT / "manifest.json").exists():
    matches = list(Path("/kaggle/input").rglob("sempointer_qwen35_frozen.bin"))
    assert len(matches) == 1, "Attach exactly one frozen bundle Dataset."
    with matches[0].open("rb") as stream:
        checksum = hashlib.file_digest(stream, "sha256").hexdigest()
    assert checksum == "ea23e4556c9fbdf8c14669fdcc078a026bf66b07e0e71728a7f7e06dbf5f22a1"
    with tarfile.open(matches[0], "r:gz") as archive:
        archive.extractall(WORK, filter="data")

manifest = runpy.run_path(str(RUNNER))["verify_bundle"](ROOT)
print("Frozen inputs verified:", len(manifest["rows"]), "main questions")

# Use a pip-less venv: Kaggle's ensurepip bootstrap previously failed.
healthy = False
if PYTHON.exists():
    probe = subprocess.run([
        str(PYTHON), "-c", "import pip, torch; assert torch.cuda.device_count() == 2"
    ], capture_output=True, text=True)
    healthy = probe.returncode == 0
if not healthy:
    venv = WORK / "venv"
    if venv.exists() or venv.is_symlink():
        venv.rename(WORK / f"venv.previous-{time.time_ns()}")
    subprocess.run([
        sys.executable, "-m", "venv", "--without-pip",
        "--system-site-packages", str(venv)
    ], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "--python", str(PYTHON),
        "install", "--no-cache-dir", "pip"
    ], check=True)

constraints = WORK / "constraints.txt"
constraints.write_text(f"torch=={torch.__version__}\n")
subprocess.run([
    str(PYTHON), "-m", "pip", "install", "--no-cache-dir",
    "--only-binary=:all:", "--constraint", str(constraints),
    "transformers==5.2.0", "accelerate==1.10.1", "bitsandbytes==0.50.2",
    "numpy==2.2.6", "ijson==3.4.0", "pyarrow==21.0.0"
], check=True)

# Download into one local directory, avoiding a duplicate weight cache.
download = r"""
import hashlib, json
from pathlib import Path
from huggingface_hub import snapshot_download
work = Path('/kaggle/working/sempointer')
manifest = json.loads((work/'frozen_bundle/manifest.json').read_text())
model = work/'models'/manifest['model_revision']
snapshot_download(manifest['model_id'], revision=manifest['model_revision'],
    local_dir=model, token=False, max_workers=2,
    allow_patterns=['*.json','*.safetensors','*.txt','*.model','*.jinja'])
with (model/'tokenizer.json').open('rb') as stream:
    assert hashlib.file_digest(stream,'sha256').hexdigest() == manifest['tokenizer_file_sha256']
index = json.loads((model/'model.safetensors.index.json').read_text())
assert all((model/name).is_file() for name in set(index['weight_map'].values()))
print('Pinned checkpoint ready:', model, flush=True)
"""
result = subprocess.run([str(PYTHON), "-u", "-c", download], check=True)

if not (WORK/"kaggle_results/config.json").exists():
    archives = ([Path(RESTORE_ARCHIVE)] if RESTORE_ARCHIVE else
                list(Path('/kaggle/input').rglob('sempointer_kaggle_results.tar.gz')))
    assert len(archives) <= 1, "Set RESTORE_ARCHIVE to the saved run you want to resume."
    if archives:
        with tarfile.open(archives[0], 'r:gz') as archive:
            assert all(m.name == 'kaggle_results' or m.name.startswith('kaggle_results/')
                       for m in archive.getmembers())
            archive.extractall(WORK, filter='data')
        print('Restored results:', archives[0])

print('Setup ready. Mode:', RUN_MODE, '| launcher allowance:', HOURS, 'hours')


In [ ]:
# A fresh interpreter uses the pinned environment and the frozen evaluator.
import signal

command = [str(PYTHON), '-u', str(RUNNER), '--mode', RUN_MODE, '--hours', str(HOURS)]
process = subprocess.Popen(command, start_new_session=True)
try:
    returncode = process.wait()
except KeyboardInterrupt:
    process.send_signal(signal.SIGINT)
    try:
        process.wait(timeout=120)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait()
    raise

print('Launcher exit code:', returncode)
archive = WORK / 'sempointer_kaggle_results.tar.gz'
if archive.exists():
    print('Download/save this result archive:', archive)
if returncode:
    for log in sorted((WORK/'kaggle_results').glob('gpu*/worker.log')):
        print('\nLast worker log lines:', log)
        print('\n'.join(log.read_text(errors='replace').splitlines()[-35:]))
    raise RuntimeError('Evaluation paused with an error. Raw checkpoints were retained; inspect the logs above.')


In [ ]:
from IPython.display import FileLink, display

projection = WORK / 'kaggle_results/pilot_projection.json'
if projection.exists():
    report = json.loads(projection.read_text())
    print('Projected remaining wall hours:', round(report['projected_wall_seconds']/3600, 2))
    print(report['caution'])

archive = WORK / 'sempointer_kaggle_results.tar.gz'
if archive.exists():
    print('Results are also available in Kaggle’s Output panel under sempointer/.')
    display(FileLink(str(archive.relative_to(Path('/kaggle/working')))))
